# Gemini ve LangChain ile LLM API'larını Çağırma Giriş 🦜🔗

Bu notebook'ta LangChain aracılığıyla LLM API'larını nasıl kullanacağınızı öğreneceksiniz. Örnek olarak Google'ın Gemini API'sını kullanacağız. Bu notebook'un sonunda, LangChain kullanarak API çağrıları yapmayı ve bunu neden yaptığımızı bileceksiniz.

## ⚙️ Kurulum

👉 Kurulum aşamasında oluşturduğumuz `.env` dosyasındaki ortam değişkenlerini yüklemek için aşağıdaki hücreyi çalıştırın:

In [1]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

True

👉 Hücrenin çıktısı "`True`" mu? Harika! Artık Gemini API ile kimlik doğrulaması yapmak için kullanılacak bir `GOOGLE_API_KEY` ortam değişkeni kurmuş olduk.

Eğer değilse, yardım isteyin.

## Basit Bir API Çağrısı Yapma

Bu notebook'ta şunların nasıl yapılacağını göstereceğiz:
1. Google'ın kendi kütüphanesini kullanarak API çağrısı yapma.
2. Aynı işlemi LangChain kullanarak yapma.

## Google Generative AI Kütüphanesini Kullanma

In [2]:
from google import genai

In [4]:
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the capital of France?",
)

`response` nesnesine bir göz atalım.

In [5]:
response.candidates[0].content.parts[0].text

'The capital of France is **Paris**.'

Gerçek cevabı nasıl alabileceğinizi görüyor musunuz?

Neyse ki, cevabı hemen almak için sadece `.text` özelliğini kullanabiliriz. Deneyin.

In [6]:
response.text

'The capital of France is **Paris**.'

Gemini cevaplarını Markdown formatında döndürür. Bunu kullanalım!

In [7]:
from IPython.display import Markdown
Markdown(response.text)

The capital of France is **Paris**.

Oluşturma parametrelerini de değiştirebilirsiniz. `google.genai` kullanarak bunu şu şekilde yaparsınız:

In [8]:
from google import genai
from google.genai import types # We need to import types for the config

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a social media post about how much you're learning about transformers.",
    config=types.GenerateContentConfig(
        max_output_tokens=200,
        temperature=1.0
    )
)

In [9]:
Markdown(response.text)

Here are a few options for a social media post about learning Transformers, playing with different tones and lengths. Choose the one that best fits your style!

**Option 1: Enthusiastic & Slightly Geeky**

> Mind. Blown. 🤯 Seriously diving deep into the world of Transformers lately, and the sheer elegance and power of these architectures are incredible. From attention mechanisms to self-attention, it's like unlocking a new level of understanding in AI. So much to learn, so much to build! #Transformers #AI #MachineLearning #DeepLearning #NLP #Tech

**Option 2: Concise & Focused**

> My brain is currently filled with attention scores and positional encodings! 🧠 Really enjoying the learning journey with Transformers. It's fascinating to see how they've revolutionized NLP and are expanding into other domains. #AIeducation #Transformers #MachineLearning

**Option 3: Slightly More Personal & Reflective**

> I've been

Harika. Ancak başka bir API denemek istediğinizi düşünün, örneğin OpenAI'nin veya Anthropic'in?

Onların dokümantasyonlarını incelemek ve tüm kodunuzu onların API'sini kullanacak şekilde yeniden yazmak zorunda kalırsınız. Tabii ki benzer olacaktır, ancak aynı olmayacaktır.

Neyse ki LangChain var!

## LangChain Kullanma 🦜🔗

Neden LangChain kullanırsınız?

1. **Model-Bağımsız Kod**

   LangChain, farklı LLM sağlayıcıları (Google, OpenAI, Anthropic, vb.) arasında minimal kod değişikliği ile geçiş yapmanızı sağlayan soyutlamalar sunar. Google API'sine doğrudan kod yazarsanız, sağlayıcı değiştirmek önemli ölçüde yeniden düzenleme gerektirir.

2. **Birleşik Arayüz**

   LangChain, altta yatan API'den bağımsız olarak farklı LLM sağlayıcıları arasında etkileşimleri standartlaştırır ve tutarlı yöntemler ile yanıt formatları sunar.

3. **Bileşenlerle Çalışabilirlik**

   LangChain'in zincir ve pipeline mimarisi, tüm alt yapıyı kendiniz halletmeden prompt, bellek ve erişim sistemlerini birleştiren karmaşık iş akışları oluşturmayı kolaylaştırır.

4. **Yerleşik Araçlar**

   LangChain, çıktı ayrıştırma, prompt şablonları ve kendiniz uygulamanız gereken diğer yardımcı araçları içerir.

[LangChain'in chat entegrasyonları listesi](https://docs.langchain.com/oss/python/integrations/chat)'ne gidin ve entegrasyon listesine bakın. Favori LLM sağlayıcınızı bulabiliyor musunuz?

Kodumuzda `chat_models.ChatGoogleGenerativeAI` kullanmak istemiyoruz çünkü bu özellikle Gemini için yapılmış. LLM'yi değiştirmek istersek, modeli başlatma şeklimizi değiştirmek zorunda kalırız. Neyse ki LangChain bir modeli başlatmak için daha genel bir yol sunar.

Gemini'yi tekrar kullanalım, ancak şimdi LangChain'in genel Chat Models'ini kullanarak.

👉 [LangChain'in "Models" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/models) sayfasına gidin ve Gemini kullanarak bir chat modelinin nasıl başlatılacağını bulun.

İpuçları:
1. Hemen "Basic Usage" bölümüne gidin.
2. Kullanmak istediğiniz modeli seçerek doğru dokümantasyonu hemen görebilirsiniz.

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

Modelin en temel kullanımı sadece `.invoke()` metodunu kullanmaktır:

In [11]:
response = model.invoke("What is the capital of France?")

Yanıta bir göz atalım. Nesnenin tüm öznitelik ve metodlarını içeren `__dict__`'ini güzel şekilde yazdırmak için `pprint()` kullanıyoruz.

In [12]:
from pprint import pprint
pprint(response.__dict__)

{'additional_kwargs': {},
 'content': 'The capital of France is **Paris**.',
 'id': 'lc_run--019c095d-63b6-73d0-ba94-68005ebc3a05-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'STOP',
                       'model_name': 'gemini-2.5-flash-lite',
                       'model_provider': 'google_genai',
                       'safety_ratings': []},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_token_details': {'cache_read': 0},
                    'input_tokens': 7,
                    'output_tokens': 8,
                    'total_tokens': 15}}


Cevabı çıkarın ve görüntüleyin. Markdown formatında olduğunu unutmayın, bu yüzden güzel görünmesini sağlayabilirsiniz.

In [13]:
Markdown(response.content)

The capital of France is **Paris**.

Modelin temperature değerini `.temperature` özniteliğine erişerek kontrol edebilirsiniz. Deneyin:

In [14]:
model.temperature, model.max_output_tokens

(0.7, None)

Modeli kullanmadan önce, özniteliklere yeni değerler atayarak oluşturma parametrelerini de ayarlayabiliriz.

Daha önce Google'ın kütüphanesini kullanarak sosyal medya gönderisi yazmak için yaptığımızın eşdeğerini kodlamaya çalışın.

> _Not_: Normal olarak modelin `max_output_tokens` değerini ayarlayabilmemiz gerekir (modeli başlatırken veya daha sonra özniteliği değiştirerek). _langchain_google_genai_'nin mevcut sürümü (4.1.1) bir [hataya](https://github.com/langchain-ai/langchain-google/issues/1454) sahip ve bu çalışmıyor. Geçici çözüm? `max_output_tokens`'ı `.invoke()` metodunun bir parametresi olarak ayarlayın.

In [15]:
# Set the maximum number of output tokens to 200
model.max_output_tokens = 200

# Set the temperature to 1.0
model.temperature = 1.0

# Generate a response with the new settings
response = model.invoke("Write a social media post about how much you're learning about transformers.")

# Display the response
Markdown(response.content)

Here are a few options for a social media post about learning about Transformers, ranging in tone and detail. Choose the one that best fits your style!

**Option 1: Enthusiastic & Simple**

> My brain is officially buzzing with all things **Transformers** lately! 🤯 This architecture is seriously mind-blowing and I'm loving diving deeper into how it all works. So much to learn, so little time (but I'm making it happen!). #Transformers #DeepLearning #AI #MachineLearning #NLP #LearningJourney

**Option 2: A Bit More Detail & Curiosity**

> I've been on a deep dive into the world of **Transformers** lately and I'm absolutely fascinated! The attention mechanism and how it handles sequential data is truly revolutionary. Feeling like my understanding of NLP is leveling up exponentially. What are your favorite Transformer resources? 👇 #AIResearch #NeuralNetworks #TransformerModel #DeepLearningAI #TechEducation

**Option 3: Humorous & Relatable**

> Pretty sure my sleep schedule is now powered by self-attention mechanisms, because I'm spending *all* my free time learning about **Transformers**! 😂 It's complex, it's powerful, and I'm completely hooked. Send coffee and more resources, please! #TransformerAI #AI #MachineLearningLife #Coding #Tech

**Option 4: Focused on a Specific Aspect (if applicable)**

> Really drilling down into the nuances of **Transformer** encoder-decoder architectures this week. The way they process information and build representations is just so elegant. Feeling a lot more confident in understanding how these models achieve such incredible results! #DeepLearning #NLP #AIModels #EncoderDecoder #TransformerArchitecture

**Option 5: Short & Sweet**

> Soaking up knowledge like a sponge when it comes to **Transformers**. This technology is changing the game! ✨ #Transformers #AI #Learning

**Remember to also consider:**

*   **Adding a relevant image or GIF:** A cool diagram of a Transformer, a relevant meme, or even just a stylized graphic can boost engagement.
*   **Tagging relevant accounts:** If you follow specific researchers or AI organizations, consider tagging them.
*   **Asking a question:** As seen in Option 2, asking a question encourages interaction.

Choose the one that resonates most with you! Happy learning!

Bunun avantajı? Bu LangChain Chat Model birçok başka API'yi destekleyebilir.

Başka bir modele geçmek için değiştirmeniz gereken tek şeyler:
1. Diğer model için bir API anahtarı alın ve kodunuzda tanımlayın.
2. Modeli başlatırken model ve sağlayıcıyı değiştirin.

### Çoklu Mesajlar

`.invoke()` fonksiyonunu sadece tek bir mesajla kullanmak biraz kısıtlayıcı.

Şu gibi birden fazla mesaj sağlayabilirsiniz:
- `SystemMessage` veya sistem mesajları: modelin nasıl davranacağını söylemek için
- `HumanMessage` veya Kullanıcı mesajları: kullanıcıdan gelen girdi
- `AIMessage` veya Asistan mesajları: modelden gelen yanıt

Bir sosyal medya yazarı yapalım.

Modele nasıl davranacağını açıklayan bir sistem mesajı göndereceğiz. Sonra kullanıcı mesajında, kendimizi sadece yazacağı konuyu vermekle sınırlayabiliriz.

Bunu nasıl yapacağınızı öğrenmek için [LangChain'in "Messages" dokümantasyonu](https://docs.langchain.com/oss/python/langchain/messages)'na bakın.

Sistem mesajı için ilhama mı ihtiyacınız var? İşte başlamanız için temel bir talimat:

```python
"""Sen Üretken AI öğrencisi için gönderiler yazan yaratıcı bir sosyal medya yazarısın.
Gönderilerinde her zaman kelime oyunu ve harekete geçirici çağrı bulunur.
Gönderilerin maksimum 200 karakter uzunluğundadır.
Her zaman emoji kullanırsın.
"""
```

In [16]:
# Import the necessary classes
from langchain_core.messages import HumanMessage, SystemMessage

# Create a list of messages
messages = [
    SystemMessage(
        """You are a creative social media writer writing posts for a Gen AI student.
        Your posts always include a pun, and a call to action.
        Your posts are maximum 200 characters long.
        You always use emojis.
        """
        ),
    HumanMessage("I'm learning about transformers."),
]

# Generate a response using the list of messages
response = model.invoke(messages)

# Display the response
Markdown(response.content)


Transformers are really *transforming* the AI game! 🤖 What's the coolest thing you've learned about them so far? Share your thoughts below! 👇

🏁 Tebrikler! Artık LangChain kullanarak çoklu mesajlarla temel prompt yazma konusunda uzmanlaştınız.